# Click simulator

Issue: #31

## Question

OTTO has no display positions, so we cannot measure position bias in it. Can we simulate impressions and clicks with position bias whose strength we know exactly, so that a bias correction (#32) can be tested against the truth?

## Data used

Input: the feature table of the training queries (`data/features/train_asof11.parquet`: 40,000 queries with 200 candidates each, real held-out labels from #22). Only the popularity feature `log_clicks` and the label are used here.

Output: `data/sim/impressions_train.parquet` and `data/sim/config.json`. Code: `notebooks/clicksim.py`.

**Model.**
- **Logging policy:** score = popularity + noise (Gaussian, standard deviation 1.0); the top **20** candidates are shown, best first. Popularity plus noise is a plausible old ranker: mostly popular items, with some randomness.
- **True relevance** `r` of an item comes from what the session really did later: 0.02 if not found later, 0.50 if clicked, 0.70 if carted, 0.90 if ordered.
- **Position-based click model:** an item at position k is clicked with probability `theta_k * r`, where `theta_k = (1 / k)^1.0` is the chance the user looks at position k. `theta_k` is the **propensity**, known exactly, and is saved with every impression.

Everything is random only through one seed, so the simulation is reproducible.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from clicksim import ETA, R_BY_LABEL, SIGMA, K, propensity, simulate_clicks

DATA = Path("data") if Path("data").exists() else Path("../data")
OUT = DATA / "sim"
OUT.mkdir(parents=True, exist_ok=True)
POOL, SEED = 200, 0

## Toy check with hand-calculated numbers

One query with 5 candidates, popularity 5, 4, 3, 2, 1 and labels 0, 3, 0, 1, 0. With no noise the order is the popularity order, and with K = 3 the shown items are candidates 0, 1, 2 at positions 1, 2, 3.

| Position | Candidate | label | r | theta | click probability |
|---|---|---|---|---|---|
| 1 | 0 | 0 | 0.02 | 1 | 0.02 |
| 2 | 1 | 3 | 0.90 | 1/2 | 0.45 |
| 3 | 2 | 0 | 0.02 | 1/3 | 0.00667 |

In [ ]:
pop = np.array([[5.0, 4.0, 3.0, 2.0, 1.0]])
lab = np.array([[0, 3, 0, 1, 0]])
one = simulate_clicks(pop, lab, seed=0, k=3, sigma=0.0)
assert one["candidate"].tolist() == [0, 1, 2] and one["position"].tolist() == [1, 2, 3]
assert np.allclose(one["propensity"], [1, 0.5, 1 / 3], atol=1e-6)
assert np.allclose(one["relevance"], [0.02, 0.9, 0.02], atol=1e-6)

# repeat many times: the average click rate at each position must match theta * r
reps = 200_000
pops = np.repeat(pop, reps, axis=0); labs = np.repeat(lab, reps, axis=0)
many = simulate_clicks(pops, labs, seed=1, k=3, sigma=0.0)
rate = many["click"].reshape(reps, 3).mean(axis=0)
expected = np.array([0.02, 0.45, 0.02 / 3])
print("empirical click rates:", rate.round(4), " expected:", expected.round(4))
assert np.allclose(rate, expected, atol=0.004)
assert propensity(1) == 1.0 and propensity(4) == 0.25
print("toy checks pass")

## Simulate for the training queries

In [ ]:
tr = pl.read_parquet(DATA / "features" / "train_asof11.parquet", columns=["session", "aid", "label", "log_clicks"])
n_q = tr.height // POOL
assert tr.height == n_q * POOL
sess = tr["session"].to_numpy().reshape(n_q, POOL)
assert (sess == sess[:, :1]).all(), "rows of one query are not contiguous"
aid = tr["aid"].to_numpy().reshape(n_q, POOL)
labels = tr["label"].to_numpy().reshape(n_q, POOL).astype(np.int64)
popularity = tr["log_clicks"].to_numpy().reshape(n_q, POOL).astype(np.float64)
print(f"{n_q:,} queries, {POOL} candidates each; label shares:", {int(k): round(float((labels == k).mean()), 5) for k in range(4)})

sim = simulate_clicks(popularity, labels, seed=SEED)
imp = pl.DataFrame({
    "session": sess[sim["query"], 0].astype(np.int64),
    "aid": aid[sim["query"], sim["candidate"]].astype(np.int32),
    "position": sim["position"],
    "click": sim["click"],
    "propensity": sim["propensity"],
    "true_relevance": sim["relevance"],
    "label": labels[sim["query"], sim["candidate"]].astype(np.int8),
})
imp.write_parquet(OUT / "impressions_train.parquet")
cfg = {"k": K, "eta": ETA, "sigma": SIGMA, "seed": SEED, "r_by_label": R_BY_LABEL.tolist(), "propensity": "theta_k = (1/k)^eta",
       "logging_policy": "popularity (log clicks before the cutoff) + Gaussian noise, top k shown", "queries": int(n_q),
       "source": "data/features/train_asof11.parquet"}
(OUT / "config.json").write_text(json.dumps(cfg, indent=2))
print(imp.height, "impressions,", int(imp["click"].sum()), "clicks (click rate", round(float(imp["click"].mean()), 4), ")")
print(imp.head(3))

## Checks

In [ ]:
EXPECTED = ["session", "aid", "position", "click", "propensity", "true_relevance"]
assert all(c in imp.columns for c in EXPECTED)
assert (imp["propensity"] > 0).all(), "propensities must be strictly positive"
assert imp.group_by("session", "aid").len()["len"].max() == 1, "an item appears twice in one list"
assert imp["position"].min() == 1 and imp["position"].max() == K
assert (imp.group_by("session").len()["len"] == K).all()
assert np.allclose(imp["propensity"].to_numpy(), propensity(imp["position"].to_numpy()), atol=1e-6)
print("columns, strictly positive propensities, one list per query, no repeated items: ok")

# 1. deterministic given the seed
h1 = hashlib.sha256(imp.write_csv().encode()).hexdigest()
again = simulate_clicks(popularity, labels, seed=SEED)
assert (again["click"] == sim["click"]).all() and (again["candidate"] == sim["candidate"]).all()
other = simulate_clicks(popularity, labels, seed=SEED + 1)
assert (other["click"] != sim["click"]).any()
print("2. same seed gives identical impressions and clicks; another seed differs. hash", h1[:16])

## The click rate by position follows the propensities

For each position, `click rate / average true relevance` is an estimate of `theta_k`. With the logging policy putting relevant items at all positions, it should be close to `(1/k)^eta`.

In [ ]:
by_pos = imp.group_by("position").agg(pl.col("click").mean().alias("click_rate"), pl.col("true_relevance").mean().alias("mean_r"), pl.len().alias("n")).sort("position")
by_pos = by_pos.with_columns((pl.col("click_rate") / pl.col("mean_r")).alias("estimated_theta"), (1.0 / pl.col("position").cast(pl.Float64) ** ETA).alias("true_theta"))
print(by_pos.select("position", "n", "click_rate", "mean_r", "estimated_theta", "true_theta"))
rel_err = ((by_pos["estimated_theta"] - by_pos["true_theta"]).abs() / by_pos["true_theta"]).to_numpy()
print("largest relative error:", rel_err.max().round(3), "at position", int(by_pos["position"][int(rel_err.argmax())]))
assert rel_err[:10].max() < 0.10, "estimated propensities for positions 1-10 are more than 10 percent off"

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(by_pos["position"], by_pos["true_theta"], "o-", label="true propensity (1/k)")
ax.plot(by_pos["position"], by_pos["estimated_theta"], "s--", label="click rate / mean relevance")
ax.set_xlabel("position"); ax.set_ylabel("examination probability"); ax.legend(); ax.set_title("Simulated click rates follow the propensities")
plt.show()

## Where the logging policy puts relevant items

This is what makes a naive click model biased: relevant items are not spread evenly over positions.

In [ ]:
share = imp.group_by("position").agg((pl.col("label") > 0).mean().alias("share_relevant")).sort("position")
print(share.head(5), "... overall share of shown items with a label:", round(float((imp["label"] > 0).mean()), 4))
naive_ctr = imp.group_by("position").agg(pl.col("click").mean().alias("ctr")).sort("position")
print("position 1 click rate is", round(float(naive_ctr["ctr"][0] / naive_ctr["ctr"][K - 1]), 1), "times the position", K, "click rate, although mean relevance only falls from", round(float(by_pos["mean_r"][0]), 4), "to", round(float(by_pos["mean_r"][K - 1]), 4))

## Result

The simulation is reproducible (same seed, identical impressions and clicks; another seed differs), and its propensities `theta_k = 1/k` are saved with the impressions. 40,000 queries give 800,000 impressions and 3,471 clicks (a click rate of 0.43 percent).

**Do the click rates follow the propensities?** Yes where there are enough clicks: the estimated propensity (click rate divided by mean true relevance) is within 1 to 4 percent of `1/k` for positions 1 to 5 and within 10 percent for positions 1 to 10 (asserted). It gets noisy deep in the list (up to 27 percent off at position 18), only because there are few clicks there (a few dozen per position): the rate at position 20 rests on about 45 clicks.

**The bias is real.** Position 1 gets 22 times the clicks of position 20 although the mean true relevance of the shown items falls only from 0.0253 to 0.0230. A model trained naively on these clicks would learn 'popular items shown at the top are relevant'. Relevant items are also slightly more common at the top (1.06 percent at position 1 vs 0.76 percent at position 5), because the logging policy uses popularity and popular items are more often relevant.

**Limits.** Clicks are rare (0.43 percent), so the IPW study in #32 will need many queries or repeated seeds to see a clear effect, and its variance will be large. The click model is the simplest one (position-based, propensity known and shared by all queries); real logs have context effects, trust bias and unknown propensities.

## What I learned

To be written by the owner of the project.